<div align="center">

<a href="https://colab.research.google.com/github/utkukose/human-computer-interaction-NB-lecture/blob/main/exams/midterm/MID2_self_service_terminal.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open in Colab"></a>

# Midterm option 2: A self-service terminal

**Human-Computer Interaction (BIL-429 İnsan-Bilgisayar Etkileşimi)**<br>Midterm project: A usability study of an interface. Starter Colab notebook

[Project page](https://github.com/utkukose/human-computer-interaction-NB-lecture/tree/main/exams/midterm) | [Course repository](https://github.com/utkukose/human-computer-interaction-NB-lecture)

**Prof. Dr. Utku Kose**

Full Professor, Department of Computer Engineering, Süleyman Demirel University, Isparta, Türkiye  
Founding Director, AI Application and Research Center (YAZEM), Süleyman Demirel University  
Head of the Computer Science Division, Department of Computer Engineering, Süleyman Demirel University  
Additional affiliations: University of North Dakota (USA), Universidad Panamericana (Mexico City, Mexico), Vel Tech University (Chennai, India)  
IEEE Senior Member, ACM Professional Member

[ORCID 0000-0002-9652-6415](https://orcid.org/0000-0002-9652-6415) | [utkukose.com](https://www.utkukose.com) | [github.com/utkukose](https://github.com/utkukose)

[utkukose@sdu.edu.tr](mailto:utkukose@sdu.edu.tr) | [utku.kose@und.edu](mailto:utku.kose@und.edu) | [ukose@up.edu.mx](mailto:ukose@up.edu.mx) | [utkukose@gmail.com](mailto:utkukose@gmail.com)

<sub>Content licensed under CC BY 4.0, code under MIT. This course is updated in line with current developments in the field. First version: November 2026.</sub>

</div>

## How to use this starter Colab notebook

Save a copy in Drive first, with File, Save a copy in Drive. Every code cell runs as it is with an example case: Buying a ticket at the ticket machine of a railway station. Replace the values marked `# TODO` with the measurements of your own interface. Then run all cells again from the top, with Runtime, Run all, and use the printed values and figures in the report. Each section names the week and the lecture section that explain its method. The last section collects the numbers for the results table of the report. Keep your own data in the cells, so that the Colab notebook documents how every number of the report was obtained.

## 0. Setup

The cell loads the libraries and creates the dictionary `RESULTS`, which collects the numbers of the report.

In [ ]:
import math

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

RESULTS = {}            # the numbers for the results table of the report

def normal_cdf(z):
    """Share of standard normal values below z; works on numbers and arrays."""
    return 0.5 * (1 + np.vectorize(math.erf)(np.asarray(z, float) / np.sqrt(2)))

print("Setup complete.")

## 1. Text size and contrast

*Lecture: [Week 2, The eye and the visual angle](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-02/lecture.html#the-eye-and-the-visual-angle) and [Week 2, Luminance and contrast](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-02/lecture.html#luminance-and-contrast)*

A text of x-height $h$ seen from the distance $d$ subtends the visual angle $\theta = 2 \arctan(h / 2d)$. Reading is fastest above a critical print size of about 0.2°, and 0.3° is comfortable. Measure the x-height of each text on the screen with a ruler, and the distance from the eyes of a standing user to the screen. The contrast ratio of the Web Content Accessibility Guidelines (WCAG) is $(L_1 + 0.05)/(L_2 + 0.05)$, with the relative luminances $L_1 \ge L_2$ of the two colours. Its level AA, the level that most laws ask for, needs 4.5:1 for normal text and 3:1 for large text.

In [ ]:
VIEW_MM = 650      # TODO: distance from the eyes to the screen (mm)
TEXTS = {          # TODO: the x-height (mm) of each text on the screen
    "title": 6.0, "list of stations": 2.6, "price": 3.2, "terms of travel": 1.6}

def visual_angle(h, d):
    """Visual angle in degrees of a height h seen from the distance d."""
    return np.degrees(2 * np.arctan(h / (2 * d)))

def height_for_angle(theta_deg, d):
    """The height that subtends theta_deg degrees at the distance d."""
    return 2 * d * np.tan(np.radians(theta_deg) / 2)

for name, h in TEXTS.items():
    a = visual_angle(h, VIEW_MM)
    verdict = ("comfortable" if a >= 0.3 else "readable" if a >= 0.2
               else "below the critical print size")
    print(f"{name:>17}: {h:.1f} mm -> {a:.3f} deg, {verdict}")
h03 = float(height_for_angle(0.3, VIEW_MM))
print(f"x-height for 0.3 deg at {VIEW_MM} mm: {h03:.2f} mm")
small = min(TEXTS, key=TEXTS.get)
RESULTS["smallest text"] = small
RESULTS["smallest visual angle (deg)"] = round(
    float(visual_angle(TEXTS[small], VIEW_MM)), 3)
RESULTS["x-height for 0.3 deg (mm)"] = round(h03, 2)

In [ ]:
PAIRS = {   # TODO: text colour, background colour and the size of the text
    "white on red button": ("#FFFFFF", "#E2001A", "large"),
    "grey price on white": ("#8A8A8A", "#FFFFFF", "text"),
    "yellow note on white": ("#F5C400", "#FFFFFF", "text"),
    "blue link on grey": ("#2F5D8A", "#E6E6E6", "text"),
}
REQUIRED = {"text": 4.5, "large": 3.0}     # WCAG 2.2, level AA

def hex_to_rgb(code):
    """'#2F5D8A' -> red, green and blue from 0 to 255."""
    code = code.lstrip("#")
    return np.array([int(code[i:i + 2], 16) for i in (0, 2, 4)])

def relative_luminance(rgb):
    """Relative luminance L of a colour given from 0 to 255."""
    c = np.asarray(rgb, float) / 255
    lin = np.where(c <= 0.04045, c / 12.92, ((c + 0.055) / 1.055) ** 2.4)
    return float(lin @ np.array([0.2126, 0.7152, 0.0722]))

def contrast_ratio(c1, c2):
    """WCAG contrast ratio of two colours given as hex codes."""
    L1, L2 = relative_luminance(hex_to_rgb(c1)), relative_luminance(hex_to_rgb(c2))
    return (max(L1, L2) + 0.05) / (min(L1, L2) + 0.05)

fails = 0
for name, (fg, bg, size) in PAIRS.items():
    cr = contrast_ratio(fg, bg)
    ok = cr >= REQUIRED[size]
    fails += not ok
    verdict = "pass" if ok else "FAIL"
    print(f"{name:>20}: {cr:5.2f}:1, needs {REQUIRED[size]}:1 -> {verdict}")
RESULTS["colour pairs failing level AA"] = f"{fails} of {len(PAIRS)}"

## 2. Choosing the destination

*Lecture: [Week 3, The Hick-Hyman law](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-03/lecture.html#the-hick-hyman-law) and [Week 3, Menus: From search to decision](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-03/lecture.html#menus-from-search-to-decision)*

A novice searches a list item by item, and the time grows with the number of items. An expert decides, and the Hick-Hyman law $T = a + b \log_2(n + 1)$ applies. The information $H = \sum_i p_i \log_2(1/p_i)$ of the choice measures how predictable the destinations are. A split menu puts the most frequent destinations on top of the full list. The experience $e$ from 0 to 1 blends the novice and the expert.

In [ ]:
# TODO: the destinations offered and the share of the sales of each
FREQUENT = {"Ankara": 0.22, "Istanbul": 0.18, "Izmir": 0.10, "Antalya": 0.08,
            "Konya": 0.06, "Eskisehir": 0.06}
N_ALL = 60                     # TODO: all destinations in the alphabetical list
P_TOP = sum(FREQUENT.values())
rest = [(1 - P_TOP) / (N_ALL - len(FREQUENT))] * (N_ALL - len(FREQUENT))
shares = list(FREQUENT.values()) + rest
T_POINT, C_NAME = 1.0, 0.25    # s to point at an item, s to read one name

def search_time(N, a=0.40, c=C_NAME):
    """Serial, self-terminating search through N items (s)."""
    return a + c * (N + 1) / 2

def hick_time(n, a=0.20, b=0.15):
    """Hick-Hyman: choice time for n equally likely options (s)."""
    return a + b * np.log2(n + 1)

def information(p):
    """Information H in bits of a choice with probabilities p."""
    p = np.asarray(p, float)
    p = p[p > 0]
    return float((p * np.log2(1 / p)).sum())

def list_time(e):
    """Alphabetical list of all destinations, for experience e."""
    return (1 - e) * search_time(N_ALL) + e * hick_time(N_ALL) + T_POINT

def split_time(e):
    """Split menu: the frequent destinations on top, then the full list."""
    top = len(FREQUENT)
    novice = (P_TOP * search_time(top)
              + (1 - P_TOP) * (0.40 + C_NAME * (top + (N_ALL + 1) / 2)))
    expert = P_TOP * hick_time(top) + (1 - P_TOP) * hick_time(N_ALL)
    return (1 - e) * novice + e * expert + T_POINT

print(f"H = {information(shares):.2f} bits, equal shares {np.log2(N_ALL):.2f} bits")
for e in (0.0, 0.5, 1.0):
    print(f"e = {e}: list {list_time(e):.2f} s, split menu {split_time(e):.2f} s")
RESULTS["destination, novice: list -> split menu (s)"] = (
    f"{list_time(0):.2f} -> {split_time(0):.2f}")
RESULTS["information of the choice (bits)"] = round(information(shares), 2)

## 3. Buttons for steady and shaking hands

*Lecture: [Week 4, Fitts' law](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-04/lecture.html#fitts-law) and [Week 4, Touch and tremor](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-04/lecture.html#touch-and-tremor)*

Fitts' law predicts the movement time (MT) $MT = a + b \log_2(D/W + 1)$ for a distance $D$ and a width $W$. Taps scatter around the aim with a spread $\sigma$, and a tremor of amplitude $A$ adds to it: $\sigma = \sqrt{\sigma_0^2 + A^2/2}$. A square button of width $W$ is hit with the probability $(2\Phi(W/2\sigma) - 1)^2$. Measure the buttons of the terminal and the distances between them.

In [ ]:
BUTTONS = {   # TODO: width W (mm) and distance D (mm) from the previous tap
    "station in the list": (9, 120), "ticket type": (14, 90),
    "pay by card": (18, 160), "confirm": (12, 60)}
USERS = {     # steady spread sigma0 (mm) and tremor amplitude A (mm)
    "typical adult": (2.0, 0.0), "older adult": (3.0, 0.0),
    "older adult with tremor": (3.0, 4.0)}

def fitts_time(D, W, a=0.10, b=0.16):
    """Predicted movement time in s, Shannon form."""
    return a + b * np.log2(D / W + 1)

def p_hit(W, sigma):
    """Probability that a tap lands on a square button of width W (mm)."""
    return float((2 * normal_cdf(W / (2 * sigma)) - 1) ** 2)

def width_for(p, sigma):
    """Width (mm) of a square button with hit probability p, by bisection."""
    lo, hi = 0.0, 100.0
    for _ in range(50):
        mid = (lo + hi) / 2
        lo, hi = (mid, hi) if p_hit(mid, sigma) < p else (lo, mid)
    return hi

rows = []
for name, (W, D) in BUTTONS.items():
    row = {"button": name, "W (mm)": W, "MT (s)": round(fitts_time(D, W), 3)}
    for user, (s0, A) in USERS.items():
        row[user] = round(p_hit(W, math.sqrt(s0 ** 2 + A ** 2 / 2)), 3)
    rows.append(row)
print(pd.DataFrame(rows).to_string(index=False))
s_tremor = math.sqrt(3.0 ** 2 + 4.0 ** 2 / 2)
need = width_for(0.95, s_tremor)
print(f"width for 95 percent hits with tremor: {need:.1f} mm")
RESULTS["width for 95 percent hits with tremor (mm)"] = round(need, 1)
worst = min(rows, key=lambda r: r["older adult with tremor"])
RESULTS["worst button with tremor"] = (
    f"{worst['button']}, hit rate {worst['older adult with tremor']}")

In [ ]:
# Figure: hit rate against the width of a button for the three user groups
W = np.linspace(4, 30, 100)
plt.figure(figsize=(6.5, 3.6))
for user, (s0, A) in USERS.items():
    s = math.sqrt(s0 ** 2 + A ** 2 / 2)
    plt.plot(W, [p_hit(w, s) for w in W], label=user)
plt.axhline(0.95, color="#C98510", ls="--", lw=1)
plt.xlabel("width of a square button W (mm)")
plt.ylabel("hit rate")
plt.legend()
plt.tight_layout()
plt.show()

## 4. The purchase with the keystroke-level model

*Lecture: [Week 6, Keystroke-level model](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-06/lecture.html#keystroke-level-model)*

The keystroke-level model (KLM) writes the purchase as a sequence of operators. M stands for a mental preparation and T for a tap on the screen. K is a key of the on-screen keyboard, and W is the wait for the terminal. Write the sequence of the current terminal and of the redesign, step by step.

In [ ]:
OPS = {"M": 1.35, "T": 0.9, "K": 0.5, "W": 2.0}   # TODO: times in s
CURRENT = ("MT" + "MTMT" + "MKKKKK" + "MT"   # start, menus, type the station
           + "MT" + "MT" + "W" + "MT" + "W")  # ticket type, pay, print
REDESIGN = "MT" + "MT" + "MT" + "W" + "MT" + "W"  # station on top, then pay

def klm_time(ops):
    """Predicted time of an operator sequence (s)."""
    return round(sum(OPS[c] for c in ops), 2)

print(f"current {klm_time(CURRENT)} s, redesign {klm_time(REDESIGN)} s, "
      f"saving {klm_time(CURRENT) - klm_time(REDESIGN):.2f} s per ticket")
RESULTS["KLM time, current (s)"] = klm_time(CURRENT)
RESULTS["KLM time, redesign (s)"] = klm_time(REDESIGN)

## 5. Results for the report

The cell collects the numbers that the sections above stored in `RESULTS`. Copy them into the results table of the report, and explain each one in the text. A value that changes the design deserves a figure.

In [ ]:
import pandas as pd

print(pd.Series(RESULTS, name="value", dtype=object).to_string())

---

<div align="center">

**Prof. Dr. Utku Kose**

Full Professor, Department of Computer Engineering, Süleyman Demirel University, Isparta, Türkiye  
Founding Director, AI Application and Research Center (YAZEM), Süleyman Demirel University  
Head of the Computer Science Division, Department of Computer Engineering, Süleyman Demirel University  
Additional affiliations: University of North Dakota (USA), Universidad Panamericana (Mexico City, Mexico), Vel Tech University (Chennai, India)  
IEEE Senior Member, ACM Professional Member

[ORCID 0000-0002-9652-6415](https://orcid.org/0000-0002-9652-6415) | [utkukose.com](https://www.utkukose.com) | [github.com/utkukose](https://github.com/utkukose)

[utkukose@sdu.edu.tr](mailto:utkukose@sdu.edu.tr) | [utku.kose@und.edu](mailto:utku.kose@und.edu) | [ukose@up.edu.mx](mailto:ukose@up.edu.mx) | [utkukose@gmail.com](mailto:utkukose@gmail.com)

This course is updated in line with current developments in the field. First version: November 2026.

</div>